# A3 handover PRB transfer demo

This notebook forces one A3 handover from gNB1 to gNB2 and checks whether the UE PRB demand moves with the UE.

Key distinction:
- `upper_demand_prbs` is the fixed persistent PRB demand attached to the UE. This should transfer exactly.
- `prbs` / `useful_prbs` are scheduler/radio outcomes after the handover. These are recalculated and may not transfer 1:1.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from global_ppo_3gnb_env import GlobalPPO3GNBEnv, normalize_slice_type

pd.set_option('display.max_colwidth', 120)


In [2]:
def ue_rows(base):
    rows = []
    for ue in base.get_all_ues():
        if not ue.connected or ue.serving_gnb is None:
            continue
        rows.append({
            'ue_id': int(ue.id),
            'serving_gnb': int(ue.serving_gnb),
            'slice': normalize_slice_type(getattr(ue, 'slice_type', 'eMBB')),
            'upper_demand_prbs': int(getattr(ue, 'upper_demand_prbs', 0)),
            'allocated_prbs': int(getattr(ue, 'prbs', 0)),
            'useful_prbs': int(getattr(ue, 'useful_prbs', 0)),
            'x': float(getattr(ue, 'x', 0.0)),
            'y': float(getattr(ue, 'y', 0.0)),
        })
    return pd.DataFrame(rows).sort_values(['serving_gnb', 'slice', 'ue_id']).reset_index(drop=True)

def cell_prb_summary(base):
    rows = []
    for gnb_id in (0, 1, 2):
        ues = [
            ue for ue in base.get_all_ues()
            if ue.connected and ue.serving_gnb is not None and int(ue.serving_gnb) == gnb_id
        ]
        rows.append({
            'gnb': f'g{gnb_id}',
            'persistent_demand_prbs': sum(int(getattr(ue, 'upper_demand_prbs', 0)) for ue in ues),
            'instant_allocated_prbs': sum(int(getattr(ue, 'prbs', 0)) for ue in ues),
            'instant_useful_prbs': sum(int(getattr(ue, 'useful_prbs', 0)) for ue in ues),
            'ue_ids': ','.join(str(int(ue.id)) for ue in ues),
        })
    return pd.DataFrame(rows)

def demand_matrix_by_slice(base):
    rows = []
    for gnb_id in (0, 1, 2):
        row = {'gnb': f'g{gnb_id}'}
        for slice_type in ('eMBB', 'URLLC', 'mMTC'):
            row[slice_type] = sum(
                int(getattr(ue, 'upper_demand_prbs', 0))
                for ue in base.get_all_ues()
                if ue.connected
                and ue.serving_gnb is not None
                and int(ue.serving_gnb) == gnb_id
                and normalize_slice_type(getattr(ue, 'slice_type', 'eMBB')) == slice_type
            )
        row['total'] = row['eMBB'] + row['URLLC'] + row['mMTC']
        rows.append(row)
    return pd.DataFrame(rows)


In [3]:
env = GlobalPPO3GNBEnv(
    seed=7,
    scenario_mode='curriculum',
    training_scenarios='mixed_overlap_with_fixed_slice_loads',
    upper_window_seconds=2.0,
    local_steps_per_global=10,
    radio_substeps=20,
    warmup_steps=0,
    safe_admission_enabled=False,
    max_handovers_per_local_step=1,
    a3_handover_cooldown_s=0.0,
    a3_min_residence_s=0.0,
)
obs, info = env.reset(seed=7)
base = env.base_env

print('handover_ttt:', base.handover_ttt)
print('a3_hysteresis_db:', base.a3_hysteresis_db)
print('max_handovers_per_step:', base.max_handovers_per_step)


handover_ttt: 3
a3_hysteresis_db: 1.0
max_handovers_per_step: 1


## Before handover

In [4]:
before_cells = cell_prb_summary(base)
before_demand_by_slice = demand_matrix_by_slice(base)
before_ues = ue_rows(base)

display(before_cells)
display(before_demand_by_slice)
display(before_ues[before_ues['serving_gnb'].eq(1)].reset_index(drop=True))


,gnb,persistent_demand_prbs,instant_allocated_prbs,instant_useful_prbs,ue_ids
0,g0,36,36,36,"8,9"
1,g1,142,100,100,"0,1,2,3,4,5,6,7"
2,g2,34,34,34,"10,11"


,gnb,eMBB,URLLC,mMTC,total
0,g0,0,0,36,36
1,g1,72,70,0,142
2,g2,0,34,0,34


,ue_id,serving_gnb,slice,upper_demand_prbs,allocated_prbs,useful_prbs,x,y
0,4,1,URLLC,18,12,12,-165.0,-15.0
1,5,1,URLLC,18,12,12,165.0,-15.0
2,6,1,URLLC,17,12,12,-165.0,45.0
3,7,1,URLLC,17,12,12,165.0,45.0
4,0,1,eMBB,18,13,13,-165.0,-30.0
5,1,1,eMBB,18,13,13,165.0,-30.0
6,2,1,eMBB,18,13,13,-165.0,30.0
7,3,1,eMBB,18,13,13,165.0,30.0


## Force only gNB1 -> gNB2 eMBB handover

All directions are set to `+6 dB` to retain/block. Then only `g1 -> g2` for eMBB is opened with `-2 dB`.

The A3 condition checked is:

`RSRP_target > RSRP_serving + offset + hysteresis`

In [5]:
for src in (0, 1, 2):
    for tgt in (0, 1, 2):
        if src == tgt:
            continue
        for slice_type in env.slice_types:
            base.set_a3_offset(src, tgt, slice_type, +6.0)

base.set_a3_offset(1, 2, 'eMBB', -2.0)

a3_rows = []
new_events = []
for local_step in range(1, 8):
    for ue in base.get_all_ues():
        if (
            ue.connected
            and ue.serving_gnb == 1
            and normalize_slice_type(getattr(ue, 'slice_type', 'eMBB')) == 'eMBB'
        ):
            rsrp_serving = base._measure_rsrp(base._get_gnb_by_id(1), ue)
            rsrp_target = base._measure_rsrp(base._get_gnb_by_id(2), ue)
            offset = base.get_a3_offset(1, 2, 'eMBB')
            threshold = rsrp_serving + offset + base.a3_hysteresis_db
            margin = rsrp_target - threshold
            a3_rows.append({
                'local_step': local_step,
                'ue_id': int(ue.id),
                'upper_demand_prbs': int(getattr(ue, 'upper_demand_prbs', 0)),
                'rsrp_serving_g1': rsrp_serving,
                'rsrp_target_g2': rsrp_target,
                'offset_db': offset,
                'hysteresis_db': base.a3_hysteresis_db,
                'a3_margin_db': margin,
                'a3_condition_true': bool(margin > 0),
            })
    before_event_count = len(base.handover_events)
    base.step(0)
    new_events = base.handover_events[before_event_count:]
    if new_events:
        break

a3_table = pd.DataFrame(a3_rows)
display(a3_table)
display(pd.DataFrame(new_events))


,local_step,ue_id,upper_demand_prbs,rsrp_serving_g1,rsrp_target_g2,offset_db,hysteresis_db,a3_margin_db,a3_condition_true
0,1,0,18,-73.803701,-89.406698,-2.0,1.0,-14.602998,False
1,1,1,18,-73.803701,-66.798179,-2.0,1.0,8.005522,True
2,1,2,18,-73.803701,-89.406698,-2.0,1.0,-14.602998,False
3,1,3,18,-73.803701,-66.798179,-2.0,1.0,8.005522,True
4,2,0,18,-73.803701,-89.406698,-2.0,1.0,-14.602998,False
5,2,1,18,-73.803701,-66.798179,-2.0,1.0,8.005522,True
6,2,2,18,-73.803701,-89.406698,-2.0,1.0,-14.602998,False
7,2,3,18,-73.803701,-66.798179,-2.0,1.0,8.005522,True
8,3,0,18,-73.803701,-89.406698,-2.0,1.0,-14.602998,False
9,3,1,18,-73.803701,-66.798179,-2.0,1.0,8.005522,True


,step,ue_id,slice_type,from_gnb,to_gnb,rsrp_serving_dbm,rsrp_target_dbm,offset_db,controller,safe_admission
0,3,1,EMBB,1,2,-73.803701,-66.798179,-2.0,MultiGNBWrapper,False


## After handover

In [6]:
after_cells = cell_prb_summary(base)
after_demand_by_slice = demand_matrix_by_slice(base)
after_ues = ue_rows(base)

display(after_cells)
display(after_demand_by_slice)
display(after_ues[after_ues['ue_id'].isin([event['ue_id'] for event in new_events])].reset_index(drop=True))


,gnb,persistent_demand_prbs,instant_allocated_prbs,instant_useful_prbs,ue_ids
0,g0,36,36,34,"8,9"
1,g1,124,100,100,"0,2,3,4,5,6,7"
2,g2,52,42,40,"1,10,11"


,gnb,eMBB,URLLC,mMTC,total
0,g0,0,0,36,36
1,g1,54,70,0,124
2,g2,18,34,0,52


,ue_id,serving_gnb,slice,upper_demand_prbs,allocated_prbs,useful_prbs,x,y
0,1,2,eMBB,18,6,6,165.0,-30.0


## Transfer check

In [7]:
comparison = before_cells.merge(after_cells, on='gnb', suffixes=('_before', '_after'))
for col in ('persistent_demand_prbs', 'instant_allocated_prbs', 'instant_useful_prbs'):
    comparison[f'{col}_delta'] = comparison[f'{col}_after'] - comparison[f'{col}_before']
display(comparison[[
    'gnb',
    'persistent_demand_prbs_before', 'persistent_demand_prbs_after', 'persistent_demand_prbs_delta',
    'instant_allocated_prbs_before', 'instant_allocated_prbs_after', 'instant_allocated_prbs_delta',
    'instant_useful_prbs_before', 'instant_useful_prbs_after', 'instant_useful_prbs_delta',
]])

moved = pd.DataFrame(new_events).iloc[0]
moved_ue = base.get_ue(int(moved['ue_id']))
print(f"Moved UE {int(moved['ue_id'])}: g{int(moved['from_gnb'])} -> g{int(moved['to_gnb'])}")
print('Moved UE persistent upper_demand_prbs:', int(getattr(moved_ue, 'upper_demand_prbs', 0)))
print('Expected persistent demand delta: g1 -18, g2 +18 for this eMBB UE')
print('Observed persistent demand deltas:', comparison.set_index('gnb')['persistent_demand_prbs_delta'].to_dict())


,gnb,persistent_demand_prbs_before,persistent_demand_prbs_after,persistent_demand_prbs_delta,instant_allocated_prbs_before,instant_allocated_prbs_after,instant_allocated_prbs_delta,instant_useful_prbs_before,instant_useful_prbs_after,instant_useful_prbs_delta
0,g0,36,36,0,36,36,0,36,34,-2
1,g1,142,124,-18,100,100,0,100,100,0
2,g2,34,52,18,34,42,8,34,40,6


Moved UE 1: g1 -> g2
Moved UE persistent upper_demand_prbs: 18
Expected persistent demand delta: g1 -18, g2 +18 for this eMBB UE
Observed persistent demand deltas: {'g0': 0, 'g1': -18, 'g2': 18}


Conclusion: the UE fixed PRB demand (`upper_demand_prbs`) transfers exactly with the handover. The instantaneous allocated/useful PRBs are recomputed by the radio scheduler after the UE arrives at gNB2, so those values can differ from the persistent PRB demand in the same step.

In [8]:
env.close()